In [4]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

model_df = pd.read_csv('f1_lap_model_data.csv')

C:\Users\bhavi\AppData\Local\Temp\ipykernel_15164\1700919368.py:5: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  model_df = pd.read_csv('f1_lap_model_data.csv')


In [5]:
track_df = pd.read_csv('track_char.csv')
track_df['EventName'] = track_df['EventName'].str.strip()
track_df['TrackDirection'] = track_df['TrackDirection'].str.strip().str.capitalize()

model_df = model_df.merge(track_df, on='EventName', how='left')

print(model_df[track_df.columns].isna().sum())

EventName            0
CircuitLength_km     0
NumCorners           0
NumDRSZones          0
TrackDirection       0
AvgSpeed_kmh         0
ElevationChange_m    0
DownforceLevel       0
dtype: int64


In [6]:
stint_median = model_df.groupby(['EventName', 'Driver', 'Stint'])['LapTime_Seconds'].transform('median')

model_df['is_anomalous_lap'] = model_df['LapTime_Seconds'] > (1.07 * stint_median)

# 1. Update clean-lap filter
model_df['is_clean_lap'] = (model_df['is_green_flag'] & 
                              ~model_df['is_pit_lap'] & 
                              (model_df['IsAccurate'] == True) &
                              ~model_df['is_anomalous_lap'])

# 2. Re-filter
model_df = model_df[model_df['is_clean_lap'] == True].copy()

# 3. Re-sort and re-derive the degradation target on the now-cleaned data
model_df = model_df.sort_values(['EventName', 'Driver', 'Stint', 'LapNumber'])
stint_baseline = model_df.groupby(['EventName', 'Driver', 'Stint'])['LapTime_Seconds'].transform('first')
model_df['DegradationDelta_Secs'] = model_df['LapTime_Seconds'] - stint_baseline

# 4. Re-check the distribution
print(model_df['DegradationDelta_Secs'].describe())
model_df = model_df[model_df['is_clean_lap'] == True].copy()
model_df = model_df.dropna(subset=['LapTime_Seconds', 'TyreLife', 'TrackTemp', 'AirTemp'])

count    64223.000000
mean        -0.881637
std          2.308675
min        -19.569000
25%         -1.991000
50%         -0.521000
75%          0.363000
max         15.030000
Name: DegradationDelta_Secs, dtype: float64


In [7]:
print(model_df[(model_df['EventName']=='Mexico City Grand Prix') & 
               (model_df['Driver']=='PIA') & 
               (model_df['LapNumber']==36)][['TrackStatus', 'IsAccurate', 'Deleted', 'DeletedReason', 'PitInTime', 'PitOutTime']])

       TrackStatus  IsAccurate  Deleted DeletedReason  PitInTime  PitOutTime
35209            1        True    False           NaN        NaN         NaN
33096            1        True    False           NaN        NaN         NaN


In [8]:
model_df = model_df.sort_values(['EventName', 'Driver', 'Stint', 'LapNumber'])

stint_baseline = model_df.groupby(['EventName', 'Driver', 'Stint'])['LapTime_Seconds'].transform('first')
model_df['DegradationDelta_Secs'] = model_df['LapTime_Seconds'] - stint_baseline

In [9]:
sample = model_df[(model_df['EventName'] == 'Bahrain Grand Prix') & (model_df['Driver'] == 'VER')]
print(sample[['LapNumber', 'Stint', 'TyreLife', 'LapTime_Seconds', 'DegradationDelta_Secs']].head(20))

       LapNumber  Stint  TyreLife  LapTime_Seconds  DegradationDelta_Secs
1263         2.0    1.0       5.0           98.403                  0.000
2861         2.0    1.0       5.0           96.296                 -2.107
1857         3.0    1.0       6.0           98.475                  0.072
3719         3.0    1.0       6.0           96.753                 -1.650
5292         3.0    1.0       6.0           98.006                 -0.397
2495         4.0    1.0       7.0           98.549                  0.146
4667         4.0    1.0       7.0           96.647                 -1.756
6268         4.0    1.0       7.0           97.976                 -0.427
3206         5.0    1.0       8.0           98.760                  0.357
5681         5.0    1.0       8.0           97.173                 -1.230
7381         5.0    1.0       8.0           98.035                 -0.368
4135         6.0    1.0       9.0           98.593                  0.190
6640         6.0    1.0       9.0     

In [10]:
features = ['TyreLife', 'Compound', 'FreshTyre', 'TrackTemp', 'AirTemp', 'Driver',
            'CircuitLength_km', 'NumCorners', 'NumDRSZones', 'TrackDirection',
            'AvgSpeed_kmh', 'ElevationChange_m', 'DownforceLevel']

X = model_df[features].copy()
y = model_df['DegradationDelta_Secs']

X = pd.get_dummies(X, columns=['Compound', 'Driver', 'TrackDirection', 'DownforceLevel'])

In [11]:

sorted_events = model_df[['RoundNumber', 'EventName']].drop_duplicates().sort_values('RoundNumber')
event_order = sorted_events['EventName'].tolist()

n_test_races = 5
train_events = event_order[:-n_test_races]
test_events = event_order[-n_test_races:]

train_mask = model_df['EventName'].isin(train_events)
test_mask = model_df['EventName'].isin(test_events)

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [12]:

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print(f"MAE: {mae:.3f} seconds")

MAE: 1.117 seconds


In [13]:
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(15))

TyreLife                 0.134656
TrackTemp                0.119408
AirTemp                  0.096853
Compound_INTERMEDIATE    0.050853
NumCorners               0.044630
ElevationChange_m        0.034905
CircuitLength_km         0.033771
AvgSpeed_kmh             0.033616
Compound_HARD            0.033094
FreshTyre                0.028024
Compound_MEDIUM          0.024868
Driver_RUS               0.020326
Driver_HUL               0.019463
Compound_SOFT            0.017809
Driver_NOR               0.017725
dtype: float64


In [14]:
import joblib
joblib.dump(model, 'tire_deg.joblib')
joblib.dump(X_train.columns, 'tire_deg_columns.joblib')

['tire_deg_columns.joblib']